# HCP Young Adult dataset analysis

## Dataset and project initialization

In [ ]:
!uv sync

In [ ]:
import glob, numpy as np, nibabel as nib, collections, pywt
import matplotlib.pyplot as plt
from pathlib import Path
from nibabel.cifti2.cifti2 import Cifti2Image
from scipy.signal import welch
from sklearn.model_selection import GroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
from scipy.stats import wilcoxon, balanced_accuracy_score, f1_score
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.svm import LinearSVC
from nilearn import plotting, image

In [ ]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
files = sorted(ROOT.glob("data/*/MNINonLinear/Results/tfMRI_*_??/*dtseries.nii"))

In [ ]:
DESIKAN = ["bankssts", "caudalanteriorcingulate", "caudalmiddlefrontal", None, "cuneus",
           "entorhinal", "fusiform", "inferiorparietal", "inferiortemporal",
           "isthmuscingulate", "lateraloccipital", "lateralorbitofrontal", "lingual",
           "medialorbitofrontal", "middletemporal", "parahippocampal", "paracentral",
           "parsopercularis", "parsorbitalis", "parstriangularis", "pericalcarine",
           "postcentral", "posteriorcingulate", "precentral", "precuneus",
           "rostralanteriorcingulate", "rostralmiddlefrontal", "superiorfrontal",
           "superiorparietal", "superiortemporal", "supramarginal", "frontalpole",
           "temporalpole", "transversetemporal", "insula"]      # None = corpus callosum

CODES = {c: f"{h}_{nm}" for k, nm in enumerate(DESIKAN) if nm
         for c, h in ((1001 + k, "L"), (2001 + k, "R"))}
CODES.update({8: "L_cerebellum", 10: "L_thalamus", 11: "L_caudate", 12: "L_putamen",
              13: "L_pallidum", 17: "L_hippocampus", 18: "L_amygdala", 26: "L_accumbens",
              28: "L_ventralDC", 47: "R_cerebellum", 49: "R_thalamus", 50: "R_caudate",
              51: "R_putamen", 52: "R_pallidum", 53: "R_hippocampus", 54: "R_amygdala",
              58: "R_accumbens", 60: "R_ventralDC", 16: "brainstem"})

CODE_LIST = sorted(CODES)
names = [CODES[c] for c in CODE_LIST]
R = len(CODE_LIST)
print(R, "regiona:", names[:3], "...", names[-3:])

In [ ]:
vol_files = sorted(ROOT.glob(
    "data/*/MNINonLinear/Results/tfMRI_*_??/*_hp0_clean_rclean_tclean.nii.gz"))
print(len(vol_files), "runova")
print(vol_files[0].parts[-5], vol_files[0].parts[-2])

In [ ]:
runs, meta, current = [], [], None

for f in vol_files:
    subject, task, enc = f.parts[-5], *f.parts[-2].split("_")[1:3]

    if subject != current:                          # parcelacija je po ispitaniku
        w = nib.load(str(ROOT / "data" / subject / "MNINonLinear/ROIs/wmparc.2.nii.gz"))
        codes = w.get_fdata().astype(np.int32).ravel()
        lab = np.zeros(codes.shape, dtype=np.int32)
        for i, c in enumerate(CODE_LIST, start=1):
            lab[codes == c] = i
        vox, lab_v = np.flatnonzero(lab), None
        lab_v = lab[vox]
        counts = np.bincount(lab_v, minlength=R + 1)[1:]
        current = subject

    img = nib.load(str(f))
    nt  = img.shape[3]
    x   = img.get_fdata(dtype=np.float32).reshape(-1, nt)[vox]
    runs.append(np.stack([np.bincount(lab_v, weights=x[:, j], minlength=R + 1)[1:] / counts
                          for j in range(nt)]).astype(np.float32))
    meta.append((subject, task, enc))
    print(f"{len(runs):3d}/{len(vol_files)}  {subject} {task}_{enc}  {runs[-1].shape}",
          flush=True)

## Pregled signala

In [ ]:
subjects  = np.array([m[0] for m in meta])
y         = np.array([m[1] for m in meta])
encodings = np.array([m[2] for m in meta])
y = np.array([m[1] for m in meta])
SUBJECTS, TASKS, ENCS = sorted(set(subjects)), sorted(set(y)), sorted(set(encodings))

print("ispitanici:", {i: s for i, s in enumerate(SUBJECTS)})
print("zadaci:    ", {i: t for i, t in enumerate(TASKS)})
print("kodiranja: ", {i: e for i, e in enumerate(ENCS)})

SUBJECT = SUBJECTS[2]        # 0 111211  1 135124  2 153126  3 192237  4 206525 ...
TASK    = TASKS[2]           # 0 EMOTION  1 GAMBLING  2 LANGUAGE  3 MOTOR
ENC     = ENCS[0]            # 4 RELATIONAL  5 SOCIAL  6 WM   |   0 LR  1 RL
IA, IB  = 0, 1               # indeksi uslova, vidi ispis "uslovi"
CZ      = 27                 # presek; MNI z = (CZ - 36) * 2 mm
DELAY   = 5.0
NFRAMES, NCOLS = 12, 4
REGION_NAME = "L_precentral"      # literal: REGION se iz njega izvodi niže

In [ ]:
ACCENT, MUTED = "#3b6fb6", "#9aa0a6"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.linewidth": 0.4, "grid.color": MUTED, "grid.alpha": 0.3,
    "axes.titlelocation": "left", "axes.titlesize": 10, "axes.labelsize": 10,
    "figure.constrained_layout.use": True,
})

In [ ]:
stem = f"tfMRI_{TASK}_{ENC}"
run  = ROOT / "data" / SUBJECT / "MNINonLinear/Results" / stem

REGION       = next(i for i, n in enumerate(names) if n.endswith(REGION_NAME))
REGION_SHORT = names[REGION].replace("CIFTI_STRUCTURE_", "")
RUNS_IDX     = np.flatnonzero((subjects == SUBJECT) & (y == TASK))

img4 = nib.load(str(run / f"{stem}_hp0_clean_rclean_tclean.nii.gz"))
TR   = float(img4.header.get_zooms()[3])
nx, ny, nz, nt = img4.shape
t    = np.arange(nt) * TR

sl       = np.asarray(img4.dataobj[:, :, CZ, :], dtype=np.float32)     # (nx, ny, nt)
mean_img = nib.load(str(run / f"{stem}_mean.nii.gz")).get_fdata()[:, :, CZ]
mask     = nib.load(str(run / "brainmask_fs.2.nii.gz")).get_fdata()[:, :, CZ] > 0
FRAMES   = np.linspace(0, nt - 1, NFRAMES).astype(int)

pct = np.where(mask[..., None],                                        # % promene
               (sl - mean_img[..., None]) / np.maximum(mean_img, 1)[..., None] * 100,
               np.nan)

CONDS = sorted(p.stem for p in (run / "EVs").glob("*.txt") if p.stem != "Sync")
COND_A, COND_B = CONDS[IA], CONDS[IB]

cond_frames = {}
for cond in (COND_A, COND_B):
    idx = set()
    for line in open(run / "EVs" / f"{cond}.txt"):
        onset, dur, _ = map(float, line.split())
        a = int(round((onset + DELAY) / TR))
        b = int(round((onset + dur + DELAY) / TR))
        idx |= set(range(max(a, 0), min(b, nt)))
    cond_frames[cond] = sorted(idx)

print(f"{SUBJECT} {stem}  nt={nt}  TR={TR}  z={CZ} (MNI z={(CZ - 36) * 2:+d} mm)  {REGION_SHORT}")
print("uslovi:", {i: c for i, c in enumerate(CONDS)})
print("frejmova:", {c: len(v) for c, v in cond_frames.items()})

In [ ]:
assert stem == f"tfMRI_{TASK}_{ENC}", "ponovo pokreni ćeliju A"

idx = np.flatnonzero((subjects == SUBJECT) & (y == TASK))        # LR i RL
fig, axes = plt.subplots(len(idx), 1, figsize=(13, 2.4 * len(idx)),
                         sharex=True, sharey=True, squeeze=False, layout="constrained")
for ax, i in zip(axes.flat, idx):
    sig = runs[i][:, REGION]
    sig = (sig - sig.mean()) / sig.std()
    ax.plot(np.arange(len(sig)) * TR, sig, lw=0.9, color=ACCENT)
    ax.set_title(f"{y[i]}_{encodings[i]}   ({len(sig)} frejmova, {len(sig) * TR:.0f} s)")
    ax.axhline(0, lw=0.6, color=MUTED, zorder=0)
axes.flat[-1].set_xlabel("vreme (s)")
fig.suptitle(f"{REGION_NAME} — {TASK}, ispitanik {SUBJECT}", x=0.01, ha="left")

In [ ]:
img4   = nib.load(str(run / f"{stem}_hp0_clean_rclean_tclean.nii.gz"))
vol    = img4.get_fdata(dtype=np.float32)                   # (91,109,91,nt), ~1 GB
mean3d = nib.load(str(run / f"{stem}_mean.nii.gz")).get_fdata()

diff3d     = vol[..., cond_frames[COND_A]].mean(-1) - vol[..., cond_frames[COND_B]].mean(-1)
contrast3d = np.where(mean3d > 0, diff3d / np.maximum(mean3d, 1) * 100, 0)
stat       = nib.Nifti1Image(contrast3d.astype(np.float32), img4.affine)

anat = ROOT / "data" / SUBJECT / "MNINonLinear/T1w_restore.2.nii.gz"
ttl  = f"{TASK}: {COND_A} − {COND_B} (%)"

plotting.plot_stat_map(stat, bg_img=str(anat), threshold=0.4, vmax=1.5,
                       display_mode="mosaic", title=ttl)


In [ ]:
i = np.flatnonzero((subjects == SUBJECT) & (y == TASK) & (encodings == ENC))[0]
Z = runs[i]
Z = ((Z - Z.mean(0)) / np.where(Z.std(0) > 0, Z.std(0), 1)).T          # (87, T), redosled = names

LEFT = [k for k, n in enumerate(names) if n.startswith("L_")]
RIGHT = [k for k, n in enumerate(names) if n.startswith("R_")]

fig, axes = plt.subplots(2, 1, figsize=(16, 13), sharex=True, layout="constrained")
for ax, idx, side in [(axes[0], LEFT, "levo"), (axes[1], RIGHT, "desno")]:
    im = ax.imshow(Z[idx], aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5,
                   extent=[0, Z.shape[1] * TR, len(idx), 0], interpolation="nearest")
    ax.set_yticks(np.arange(len(idx)) + 0.5)
    ax.set_yticklabels([names[k] for k in idx], fontsize=7)
    ax.set_xlabel("vreme (s)")
    ax.set_title(side)
    ax.grid(False)
    axes[0].set_xlabel("")          # samo donji panel nosi oznaku
    for f in cond_frames[COND_A]:
        ax.axvline(f * TR, color="k", lw=0.3, alpha=0.15)
fig.colorbar(im, ax=axes, shrink=0.6, label="z")
fig.suptitle(f"{SUBJECT} — {stem}, z-score po regionu", x=0.01, ha="left")

In [ ]:
assert stem == f"tfMRI_{TASK}_{ENC}", "ponovo pokreni ćeliju A"

sd = sl.std(axis=-1)
vx, vy = np.unravel_index(np.argmax(np.where(mask, sd, 0)), sd.shape)

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, sig, title in [(axes[0], sl[vx, vy, :], f"voksel ({vx}, {vy}, {CZ})"),
                       (axes[1], sl[mask].mean(axis=0), f"prosek preseka z={CZ}")]:
    ax.plot(t, (sig - sig.mean()) / sig.std(), lw=0.9, color=ACCENT)
    ax.set_title(title)
    ax.axhline(0, lw=0.6, color=MUTED, zorder=0)
axes[-1].set_xlabel("vreme (s)")
fig.suptitle(f"{SUBJECT} — {stem}", x=0.01, ha="left")

In [ ]:
NPERSEG = 128
FS, NYQ = 1 / TR, 1 / (2 * TR)

psd = {}
for task in TASKS:
    acc = []
    for i in np.flatnonzero(y == task):
        sig = runs[i][:, REGION]
        f, p = welch((sig - sig.mean()) / sig.std(), fs=FS, nperseg=NPERSEG)
        acc.append(p)
    psd[task] = np.mean(acc, axis=0)
grand = np.mean(list(psd.values()), axis=0)

NR = -(-len(TASKS) // NCOLS)
fig, axes = plt.subplots(NR, NCOLS, figsize=(3.3 * NCOLS, 2.9 * NR),
                         sharex=True, sharey=True)
for ax, task in zip(axes.flat, TASKS):
    for l in range(1, 6):
        ax.axvline(NYQ / 2 ** l, lw=0.5, color=MUTED, alpha=0.6, zorder=0)
    ax.semilogy(f, grand, lw=1.0, color=MUTED, zorder=1)
    ax.semilogy(f, psd[task], lw=1.5, color=ACCENT, zorder=2)
    ax.set_title(f"{task}  (n={np.sum(y == task)})")
for j, ax in enumerate(axes.flat):
    if j >= len(TASKS):
        ax.set_visible(False)
    elif j + NCOLS >= len(TASKS):
        ax.set_xlabel("frekvencija (Hz)")
fig.suptitle(f"Spektar po zadatku — {REGION_NAME} (sivo = prosek svih)", x=0.01, ha="left")

## Prozori i podela

In [ ]:
WIN, STRIDE = 176, 88                      # 176 = najkraći run; 50% preklapanja
N_TRAIN, N_VAL, N_TEST, SEED = 6, 2, 2, 0

Xw, yw, gw, srcw = [], [], [], []
for i, r in enumerate(runs):
    starts = list(range(0, r.shape[0] - WIN + 1, STRIDE))
    if starts[-1] != r.shape[0] - WIN:
        starts.append(r.shape[0] - WIN)                    # rep runa
    for s in starts:
        seg = r[s:s + WIN]
        Xw.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
        yw.append(y[i]); gw.append(subjects[i]); srcw.append((i, s))

Xw   = np.stack(Xw).astype(np.float32)                     # (357, 176, 87)
yw, gw, srcw = np.array(yw), np.array(gw), np.array(srcw)

perm = np.random.default_rng(SEED).permutation(np.array(SUBJECTS))
SUBJ = {"train": sorted(perm[:N_TRAIN]),
        "val":   sorted(perm[N_TRAIN:N_TRAIN + N_VAL]),
        "test":  sorted(perm[N_TRAIN + N_VAL:])}
IDX  = {k: np.flatnonzero(np.isin(gw, s)) for k, s in SUBJ.items()}

LOSO      = list(LeaveOneGroupOut().split(Xw, yw, gw))
LOSO_SUBJ = [gw[te][0] for _, te in LOSO]

assert sum(len(v) for v in IDX.values()) == len(Xw)
print(f"prozora: {len(Xw)}  oblik {Xw.shape}")
for k in ("train", "val", "test"):
    print(f"  {k:5s} {list(SUBJ[k])} -> {len(IDX[k]):3d}")
print(f"LOSO: {len(LOSO)} foldova")
# Runovi različite dužine daju različit broj prozora, pa skup nije uravnotežen:
# trivijalna granica je većinska klasa, ne 1/7.
CNT    = collections.Counter(yw)
CHANCE = max(CNT.values()) / len(yw)
print("prozora po klasi:", dict(sorted(CNT.items(), key=lambda t: -t[1])))
print(f"uniformno pogađanje {1 / len(CNT):.3f}   većinska klasa {CHANCE:.3f}")

## Obeležja

In [ ]:
FS = 1 / TR
f, P = welch(Xw, fs=FS, nperseg=WIN, axis=1)          # (n, nf, 87)
p = P / P.sum(axis=1, keepdims=True)                  # relativna snaga
NLEV = 4                                    # isti broj nivoa kao DWT
NYQ  = 1 / (2 * TR)
edges = [f[1]] + [NYQ / 2 ** k for k in range(NLEV, 0, -1)] + [NYQ]
BANDS = list(zip(edges[:-1], edges[1:]))
BAND_NAMES = [f"A{NLEV}"] + [f"D{k}" for k in range(NLEV, 0, -1)]

for nm, (lo, hi) in zip(BAND_NAMES, BANDS):
    print(f"  {nm:3s}  {lo:.4f}–{hi:.4f} Hz")

In [ ]:
blocks = [p[:, (f >= lo) & (f < hi), :].sum(axis=1) for lo, hi in BANDS]
blocks += [-(p * np.log(p + 1e-12)).sum(axis=1),      # spektralna entropija
           (f[None, :, None] * p).sum(axis=1),        # spektralno težište
           (Xw[:, :-1] * Xw[:, 1:]).mean(axis=1),     # autokorelacija lag 1
           (Xw[:, :-5] * Xw[:, 5:]).mean(axis=1)]     # autokorelacija lag 5

BLOCK_NAMES = [f"band_{nm}" for nm in BAND_NAMES] + ["entropy", "centroid", "ac1", "ac5"]
FEAT_NAMES  = [f"{b}:{r}" for b in BLOCK_NAMES for r in names]

F_std = np.concatenate(blocks, axis=1).astype(np.float32)

print(f"F_std: {F_std.shape}  ({len(BLOCK_NAMES)} tipa × {len(names)} regiona)")
print(f"NaN/inf: {(~np.isfinite(F_std)).sum()}")

In [ ]:
WAVELET = "db4"                       # ili "sym4"
L   = pywt.dwt_max_level(WIN, pywt.Wavelet(WAVELET).dec_len)
NYQ = 1 / (2 * TR)

coeffs = pywt.wavedec(Xw, WAVELET, level=L, axis=1)      # [cA_L, cD_L, ..., cD_1]
SUB    = [f"A{L}"] + [f"D{k}" for k in range(L, 0, -1)]

print(f"{WAVELET}, L = {L}   (filtar {pywt.Wavelet(WAVELET).dec_len} koef., prozor {WIN})")
for s, c in zip(SUB, coeffs):
    k = int(s[1:])
    lo, hi = (0, NYQ / 2 ** L) if s[0] == "A" else (NYQ / 2 ** k, NYQ / 2 ** (k - 1))
    print(f"  {s:3s}  {c.shape[1]:3d} koeficijenata   {lo:.3f}–{hi:.3f} Hz")

In [ ]:
energy = [(c ** 2).sum(axis=1) for c in coeffs]              # (n, 87) po podopsegu
total  = np.maximum(np.sum(energy, axis=0), 1e-12)
rel    = [e / total for e in energy]                          # relativna energija

ent = []
for c in coeffs:                                              # entropija koeficijenata
    q = c ** 2
    q = q / np.maximum(q.sum(axis=1, keepdims=True), 1e-12)
    ent.append(-(q * np.log(q + 1e-12)).sum(axis=1))

F_dwt     = np.concatenate(rel + ent, axis=1).astype(np.float32)
DWT_NAMES = [f"{p}_{s}:{r}" for p in ("E", "H") for s in SUB for r in names]
F_all     = np.concatenate([F_std, F_dwt], axis=1)
ALL_NAMES = FEAT_NAMES + DWT_NAMES

print(f"standardna {F_std.shape}   DWT {F_dwt.shape}   zajedno {F_all.shape}")
print(f"NaN/inf: {(~np.isfinite(F_dwt)).sum()}")

## Obrada signala

In [ ]:
# Fajlovi su hp0_clean_rclean_tclean: hp0 znači da visokopropusnog filtriranja praktično
# nema, samo uklanjanje srednje vrednosti. Spori drift skenera je zato i dalje u signalu,
# a pada u A4 opseg (0–0.043 Hz) — isti onaj u kom su najdiskriminativnija obeležja.
from scipy.signal import butter, filtfilt, detrend

HP_HZ  = 0.01
bb, aa = butter(2, HP_HZ / (0.5 / TR), btype="highpass")

vd, vg = [], []
for rn in runs:
    z = rn.astype(np.float64) - rn.mean(0)
    vd.append(np.var(z - detrend(z, axis=0)) / np.var(z))
    vg.append(np.var(np.repeat(z.mean(1, keepdims=True), z.shape[1], 1)) / np.var(z))
print(f"udeo varijanse — linearni trend {np.mean(vd):.3f} ± {np.std(vd):.3f}   "
      f"globalni signal {np.mean(vg):.3f} ± {np.std(vg):.3f}")

acc = {"kako jeste": [], "detrend": [], f"detrend + hp {HP_HZ}": []}
for rn in runs:
    z  = rn.astype(np.float64) - rn.mean(0)
    sd = np.maximum(z.std(0), 1e-9)                     # ista normalizacija za sve tri
    zd = detrend(z, axis=0)
    for k, zz in (("kako jeste", z), ("detrend", zd),
                  (f"detrend + hp {HP_HZ}", filtfilt(bb, aa, zd, axis=0))):
        fp, PP = welch(zz / sd, fs=1 / TR, nperseg=128, axis=0)
        acc[k].append(PP.mean(1))

fig, ax = plt.subplots(figsize=(7.5, 4.2), layout="constrained")
for (k, v), col in zip(acc.items(), (MUTED, ACCENT, "#b6553b")):
    ax.loglog(fp, np.mean(v, axis=0), lw=1.6, color=col, label=k)
for lo, hi in BANDS:
    ax.axvline(lo, lw=0.5, color=MUTED, alpha=0.6, zorder=0)
ax.axvline(HP_HZ, lw=1, color="#b6553b", ls=":", zorder=0)
ax.set_xlabel("frekvencija (Hz)"); ax.set_ylabel("srednja gustina snage")
ax.legend(frameon=False, fontsize=9)
ax.set_title("Spektar pre i posle uklanjanja drifta (sive linije: granice opsega)")

In [ ]:
# Odluka se donosi po `val` koloni. LOSO kolona opisuje rasipanje po ispitanicima i NE sme
# da bira, jer bi se biralo na skupu na kom se izveštava.
VAR = {"0 kako jeste":        dict(det=False, hp=False, gsr=False),
       "1 detrend":           dict(det=True,  hp=False, gsr=False),
       "2 detrend + hp":      dict(det=True,  hp=True,  gsr=False),
       "3 + globalni signal": dict(det=True,  hp=True,  gsr=True),
       "4 samo globalni":     dict(det=False, hp=False, gsr=True)}

print(f"{'obrada':22s} {'val':>6s} {'LOSO (A standardna)':>21s}")
prep = {}
for tag, cfg in VAR.items():
    proc = []
    for rn in runs:
        z = rn.astype(np.float64) - rn.mean(0)
        if cfg["det"]:
            z = detrend(z, axis=0, type="linear")
        if cfg["hp"]:
            z = filtfilt(bb, aa, z, axis=0)
        if cfg["gsr"]:
            z = z - z.mean(1, keepdims=True)            # globalni signal po trenutku
        proc.append(z.astype(np.float32))

    Xv, yv, gv = [], [], []
    for i, rn in enumerate(proc):
        st = list(range(0, rn.shape[0] - WIN + 1, STRIDE))
        if st[-1] != rn.shape[0] - WIN:
            st.append(rn.shape[0] - WIN)
        for s in st:
            seg = rn[s:s + WIN]
            Xv.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
            yv.append(y[i]); gv.append(subjects[i])
    Xv, yv, gv = np.stack(Xv).astype(np.float32), np.array(yv), np.array(gv)

    ff, PP = welch(Xv, fs=1 / TR, nperseg=WIN, axis=1)
    pp = PP / PP.sum(axis=1, keepdims=True)
    bk = [pp[:, (ff >= lo) & (ff < hi), :].sum(axis=1) for lo, hi in BANDS]
    bk += [-(pp * np.log(pp + 1e-12)).sum(axis=1), (ff[None, :, None] * pp).sum(axis=1),
           (Xv[:, :-1] * Xv[:, 1:]).mean(axis=1), (Xv[:, :-5] * Xv[:, 5:]).mean(axis=1)]
    Fp = np.concatenate(bk, axis=1).astype(np.float32)

    lp = list(GroupKFold(n_splits=min(10, len(set(gv)))).split(Xv, yv, gv))
    a  = np.array([GaussianNB().fit(Fp[tr], yv[tr]).score(Fp[te], yv[te]) for tr, te in lp])
    v  = GaussianNB().fit(Fp[IDX["train"]], yv[IDX["train"]]).score(Fp[IDX["val"]], yv[IDX["val"]])
    prep[tag] = a
    print(f"{tag:22s} {v:6.3f} {a.mean():.3f} ± {a.std():.3f}", flush=True)

In [ ]:
for tag in list(prep)[1:]:
    d = prep[tag] - prep["0 kako jeste"]
    p = 1.0 if np.allclose(d, 0) else wilcoxon(prep[tag], prep["0 kako jeste"],
                                               zero_method="zsplit").pvalue
    print(f"{tag:22s} − kako jeste: {d.mean():+.3f} ± {d.std():.3f}   "
          f"pozitivnih {(d > 0).sum()}/{len(d)}   p={p:.3f}")

In [ ]:
# Prag od 0.01 Hz je ispod frekvencijske rezolucije prozora (1/126.7 s = 0.0079 Hz), pa nije
# ni mogao da deluje. Ovde se prag podiže tako da zaista zaseca u A4 opseg (0–0.043 Hz).
# Gleda se i tačnost i F-vrednost samih band_A4 obeležja: ako F padne a tačnost ostane,
# A4 je bio drift i ostala obeležja ga nadoknađuju; ako oba ostanu, A4 je neuralnog porekla.
print(f"{'prag':>7s} {'val':>6s} {'LOSO':>16s} {'F(band_A4)':>11s} {'udeo A4 snage':>14s}")
for hz in (None, 0.01, 0.02, 0.04, 0.08):
    proc = []
    for rn in runs:
        z = detrend(rn.astype(np.float64) - rn.mean(0), axis=0, type="linear")
        if hz:
            b2, a2 = butter(2, hz / (0.5 / TR), btype="highpass")
            z = filtfilt(b2, a2, z, axis=0)
        proc.append(z.astype(np.float32))

    Xv, yv, gv = [], [], []
    for i, rn in enumerate(proc):
        st = list(range(0, rn.shape[0] - WIN + 1, STRIDE))
        if st[-1] != rn.shape[0] - WIN:
            st.append(rn.shape[0] - WIN)
        for s in st:
            seg = rn[s:s + WIN]
            Xv.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
            yv.append(y[i]); gv.append(subjects[i])
    Xv, yv, gv = np.stack(Xv).astype(np.float32), np.array(yv), np.array(gv)

    ff, PP = welch(Xv, fs=1 / TR, nperseg=WIN, axis=1)
    pp = PP / PP.sum(axis=1, keepdims=True)
    bk = [pp[:, (ff >= lo) & (ff < hi), :].sum(axis=1) for lo, hi in BANDS]
    bk += [-(pp * np.log(pp + 1e-12)).sum(axis=1), (ff[None, :, None] * pp).sum(axis=1),
           (Xv[:, :-1] * Xv[:, 1:]).mean(axis=1), (Xv[:, :-5] * Xv[:, 5:]).mean(axis=1)]
    Fp = np.concatenate(bk, axis=1).astype(np.float32)

    lp = list(GroupKFold(n_splits=min(10, len(set(gv)))).split(Xv, yv, gv))
    a  = np.array([GaussianNB().fit(Fp[tr], yv[tr]).score(Fp[te], yv[te]) for tr, te in lp])
    v  = GaussianNB().fit(Fp[IDX["train"]], yv[IDX["train"]]).score(Fp[IDX["val"]], yv[IDX["val"]])
    Fs_, _ = f_classif(Fp[IDX["train"]], yv[IDX["train"]])
    nA4 = bk[0].shape[1]                                   # band_A4 je prvi blok
    print(f"{str(hz):>7s} {v:6.3f} {a.mean():6.3f} ± {a.std():.3f} "
          f"{np.nan_to_num(Fs_)[:nA4].mean():11.1f} {bk[0].mean():14.4f}", flush=True)

## Analiza obeležja

In [ ]:
# Interpretacija obeležja, a ne samo njihova upotreba u klasifikatoru. Tri pitanja:
#   1) koliko su DWT obeležja zaista NOVA u odnosu na standardna,
#   2) koji tip obeležja nosi diskriminativnu moć,
#   3) koji regioni je nose, i da li to ima smisla za ovaj skup zadataka.
# Sve se računa isključivo na trening skupu, da interpretacija ne gleda val ni test.
Fv, _ = f_classif(F_all[IDX["train"]], yw[IDX["train"]])
Fv    = np.nan_to_num(Fv)
pref  = np.array([n.split(":")[0] for n in ALL_NAMES])
reg   = np.array([n.split(":")[1] for n in ALL_NAMES])

In [ ]:
# 1) Redundantnost: granice opsega u standardnom skupu su diadske (NYQ/2^k), dakle iste kao
#    DWT podopsezi, pa band_s (relativna snaga iz Welch-a) i E_s (relativna energija
#    koeficijenata) mere istu fizičku veličinu, samo drugim postupkom.
print("Redundantnost standardnog i DWT dela, po podopsegu")
print(f"{'podopseg':9s} {'opseg (Hz)':>14s} {'r(band, E)':>11s} {'F(band)':>9s} {'F(E)':>8s}")
r_red = []
for si, s in enumerate(SUB):
    rr = np.array([np.corrcoef(blocks[si][:, j], rel[si][:, j])[0, 1] for j in range(len(names))])
    r_red.append(np.nanmean(rr))
    k = int(s[1:])
    lo, hi = (0, NYQ / 2 ** k) if s[0] == "A" else (NYQ / 2 ** k, NYQ / 2 ** (k - 1))
    print(f"{s:9s} {f'{lo:.3f}-{hi:.3f}':>14s} {r_red[-1]:11.3f} "
          f"{Fv[pref == f'band_{s}'].mean():9.1f} {Fv[pref == f'E_{s}'].mean():8.1f}")

In [ ]:
# 2) i 3) Diskriminativnost po tipu obeležja i po regionu.
coarse = np.array(["band" if p.startswith("band_") else "E" if p.startswith("E_")
                   else "H" if p.startswith("H_") else p for p in pref])
by_kind = dict(sorted({k: Fv[coarse == k].mean() for k in dict.fromkeys(coarse)}.items(),
                      key=lambda t: t[1]))
by_reg  = dict(sorted({r: Fv[reg == r].mean() for r in dict.fromkeys(reg)}.items(),
                      key=lambda t: t[1])[-15:])
print(f"\nprosečan F po tipu obeležja: "
      + "  ".join(f"{k}={v:.1f}" for k, v in reversed(by_kind.items())))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), layout="constrained")
axes[0].bar(range(len(SUB)), r_red, color=ACCENT)
axes[0].set_xticks(range(len(SUB))); axes[0].set_xticklabels(SUB)
axes[0].set_ylim(0, 1); axes[0].set_ylabel("r(band, E)")
axes[0].set_title("1 — koliko DWT energija ponavlja snagu u opsegu")

axes[1].barh(list(by_kind), list(by_kind.values()), color=ACCENT)
axes[1].set_xlabel("prosečan F"); axes[1].set_title("2 — diskriminativnost po tipu obeležja")

axes[2].barh(list(by_reg), list(by_reg.values()), color=ACCENT)
axes[2].tick_params(axis="y", labelsize=7)
axes[2].set_xlabel("prosečan F"); axes[2].set_title("3 — 15 najinformativnijih regiona")
fig.suptitle("Analiza obeležja (samo trening skup)", x=0.01, ha="left")

In [ ]:
Fv, pv = f_classif(F_all[IDX["train"]], yw[IDX["train"]])
Fv = np.nan_to_num(Fv)
order = np.argsort(-Fv)

print("najdiskriminativnijih 25:")
for j in order[:25]:
    print(f"  {ALL_NAMES[j]:34s} F={Fv[j]:7.2f}")

# udeo po tipu obeležja i po podopsegu
import re
kind = np.array([n.split(":")[0].split("_")[0] for n in ALL_NAMES])
top  = order[:200]
print("\nudeo u top 200 po tipu:", dict(sorted(collections.Counter(kind[top]).items(),
                                               key=lambda t: -t[1])))
reg  = np.array([n.split(":")[1] for n in ALL_NAMES])
print("najčešći regioni:", collections.Counter(reg[top]).most_common(10))

In [ ]:
# Zašto B pri k=50 gubi: DWT obeležja su bliski duplikati band_* obeležja, pa pri uskom
# budžetu istiskuju komplementarne tipove (ac1, ac5, centroid, entropy) umesto da ih dopune.
for k in (25, 50, 100, 400):
    print(f"\nk = {k}")
    for tag, F, NM in [("A standardna", F_std, FEAT_NAMES), ("B standardna+DWT", F_all, ALL_NAMES)]:
        cnt = collections.Counter()
        for tr, te in LOSO:
            Fs_, _ = f_classif(F[tr], yw[tr])
            sub = np.argsort(-np.nan_to_num(Fs_))[:k]
            cnt.update("band" if (p := NM[j].split(":")[0]).startswith("band_")
                       else "E" if p.startswith("E_") else "H" if p.startswith("H_") else p
                       for j in sub)
        tot = sum(cnt.values())
        print(f"  {tag:18s} " + "  ".join(f"{t}={100 * c / tot:.0f}%" for t, c in cnt.most_common()))

In [ ]:
# Rad [01] koristi kubne Battle-Lemarié talasiće, kojih u pywt nema, pa izbor db4 mora da se
# obrazloži. Broj nivoa zavisi od dužine filtra, pa se sa talasićem menja i broj obeležja.
print(f"{'talasić':8s} {'dec_len':>8s} {'L':>3s} {'DWT obeležja':>13s}   {'LOSO (skup B)':>16s}")
for wav in ("db2", "db4", "db8", "sym4", "sym8", "coif2"):
    Lw = pywt.dwt_max_level(WIN, pywt.Wavelet(wav).dec_len)
    cf = pywt.wavedec(Xw, wav, level=Lw, axis=1)
    en  = [(c ** 2).sum(axis=1) for c in cf]
    tot = np.maximum(np.sum(en, axis=0), 1e-12)
    rl  = [e / tot for e in en]
    et  = []
    for c in cf:
        q = c ** 2
        q = q / np.maximum(q.sum(axis=1, keepdims=True), 1e-12)
        et.append(-(q * np.log(q + 1e-12)).sum(axis=1))
    Fd = np.concatenate(rl + et, axis=1).astype(np.float32)
    Fa = np.concatenate([F_std, Fd], axis=1)
    a  = np.array([GaussianNB().fit(Fa[tr], yw[tr]).score(Fa[te], yw[te]) for tr, te in LOSO])
    print(f"{wav:8s} {pywt.Wavelet(wav).dec_len:8d} {Lw:3d} {Fd.shape[1]:13d}   "
          f"{a.mean():.3f} ± {a.std():.3f}", flush=True)

In [ ]:
# Battle-Lemarié tačno, u frekvencijskom domenu. Filtar je beskonačan sa eksponencijalnim
# opadanjem, pa filtarska banka traži skraćivanje — a na prozoru od 176 uzoraka skraćenje
# koje daje L=4 nosi grešku rekonstrukcije od 30% standardne devijacije. Za relativnu
# energiju po podopsegu banka nije ni potrebna: po Parsevalu je energija podopsega
# Σ_ω |X(ω)|²·|filtar(ω)|², a |H(ω)| i |G(ω)| ortonormalizovanog splajna se znaju tačno.
# Bitno je jer je A4 najdiskriminativniji opseg, a db4 ga na ovako kratkom prozoru precenjuje.
NSPL = 3                                        # kubni splajn, kao u radu [01]
om   = 2 * np.pi * np.fft.rfftfreq(WIN)

wg, Sg = np.linspace(0, 2 * np.pi, 1 << 14, endpoint=False), 0.0
for kk in range(-60, 61):                       # S(ω) = Σ_k |N̂(ω+2πk)|², 2π-periodična
    xx = wg + 2 * np.pi * kk
    with np.errstate(invalid="ignore", divide="ignore"):
        tt = (np.sin(xx / 2) / (xx / 2)) ** (2 * (NSPL + 1))
    Sg = Sg + np.where(np.abs(xx) < 1e-12, 1.0, tt)
Sf = lambda v: np.interp(np.mod(v, 2 * np.pi), wg, Sg, period=2 * np.pi)

Pw, low = [], np.ones_like(om)                  # kaskada: |A_l|² i |D_l|²
for l in range(1, L + 1):
    aa_ = 2.0 ** (l - 1) * om
    H2 = 2 * np.abs(np.cos(aa_ / 2)) ** (2 * (NSPL + 1)) * Sf(aa_) / Sf(2 * aa_)
    G2 = 2 * np.abs(np.cos((aa_ + np.pi) / 2)) ** (2 * (NSPL + 1)) * Sf(aa_ + np.pi) / Sf(2 * aa_)
    Pw.append(low * G2 / 2)
    low = low * H2 / 2
Pw = [low] + Pw[::-1]                           # [A_L, D_L, ..., D_1] — isti redosled kao SUB
print(f"provera potpunosti banke: max |Σ|filtar|² − 1| = {np.abs(np.sum(Pw, 0) - 1).max():.2e}")

Xf   = np.fft.rfft(Xw, axis=1)                  # (n, WIN//2+1, R)
wgt  = np.ones(len(om)); wgt[1:-1] = 2.0        # rfft: unutrašnji binovi se broje dvaput
Ebl  = np.stack([(wgt[None, :, None] * np.abs(Xf) ** 2 * pw[None, :, None]).sum(1) for pw in Pw])
Ebl  = Ebl / np.maximum(Ebl.sum(0, keepdims=True), 1e-12)
F_bl = np.concatenate(list(Ebl), axis=1).astype(np.float32)

print(f"\n{'podopseg':9s} {'BL (tačno)':>11s} {'db4':>8s} {'idealno':>9s} {'r':>7s}")
for si, s in enumerate(SUB):
    ideal = 2.0 ** -(L if s[0] == "A" else int(s[1:]))
    print(f"{s:9s} {Ebl[si].mean():11.4f} {rel[si].mean():8.4f} {ideal:9.4f} "
          f"{np.corrcoef(Ebl[si].ravel(), rel[si].ravel())[0, 1]:7.3f}")

F_blall = np.concatenate([F_std, F_bl], axis=1)
print()
for tag, F in [("A standardna", F_std), ("B standardna+DWT (db4)", F_all),
               ("B' standardna+DWT (BL)", F_blall)]:
    a = np.array([GaussianNB().fit(F[tr], yw[tr]).score(F[te], yw[te]) for tr, te in LOSO])
    print(f"{tag:24s} {F.shape[1]:5d} obeležja   LOSO {a.mean():.3f} ± {a.std():.3f}")

## Model

In [ ]:
# Trodelna podela kako zadatak traži: obučavanje na train, svaki izbor na val, a test se
# dodiruje JEDNOM, na kraju, i ne učestvuje ni u jednoj odluci. LOSO stoji uz to kao procena
# rasipanja po ispitanicima — koristi sve ispitanike, pa nije zamena za test skup nego dopuna.
print("train", [str(s) for s in SUBJ["train"]])
print("val  ", [str(s) for s in SUBJ["val"]])
print("test ", [str(s) for s in SUBJ["test"]], "\n")

print(f"{'skup':20s} {'obeležja':>8s} {'val':>6s} {'test':>6s}   {'LOSO':>14s}   opseg")
res_split = {}
for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all), ("C samo DWT", F_dwt)]:
    m = GaussianNB().fit(F[IDX["train"]], yw[IDX["train"]])
    v = m.score(F[IDX["val"]], yw[IDX["val"]])
    t = m.score(F[IDX["test"]], yw[IDX["test"]])
    a = np.array([GaussianNB().fit(F[tr], yw[tr]).score(F[te], yw[te]) for tr, te in LOSO])
    res_split[tag] = {"val": v, "test": t, "loso": a,
                      "po": [m.score(F[np.flatnonzero(gw == s)], yw[np.flatnonzero(gw == s)])
                             for s in SUBJ["test"]]}
    print(f"{tag:20s} {F.shape[1]:8d} {v:6.3f} {t:6.3f}   "
          f"{a.mean():6.3f} ± {a.std():.3f}   {a.min():.2f}–{a.max():.2f}")
print(f"{'većinska klasa':20s} {'':8s} {'':>6s} {CHANCE:6.3f}")

print("\ntačnost po test ispitaniku (zbirni broj sa dva ispitanika ne govori o rasipanju):")
for tag in res_split:
    print(f"  {tag:20s} " + "   ".join(f"{str(s)}: {v:.3f}"
          for s, v in zip(SUBJ["test"], res_split[tag]["po"])))

In [ ]:
# Test skup su dva ispitanika, a tačnost po ispitaniku ide od 0.12 do 0.97, pa konačan broj
# zavisi od toga koja dva su ispala. Ista podela 6/2/2 ponavlja se preko 20 semena; gleda se
# raspodela tačnosti na testu i raspodela razlike B−A. Podela SEED = 0 ostaje ona prijavljena
# u radu — ovo je procena njene krhkosti, ne novi izbor.
SEEDS = range(20)
rows = {"A standardna": [], "B standardna+DWT": [], "B − A": []}
for sd in SEEDS:
    pm_ = np.random.default_rng(sd).permutation(np.array(SUBJECTS))
    tr_ = np.flatnonzero(np.isin(gw, pm_[:N_TRAIN]))
    te_ = np.flatnonzero(np.isin(gw, pm_[N_TRAIN + N_VAL:]))
    sc = []
    for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all)]:
        s_ = GaussianNB().fit(F[tr_], yw[tr_]).score(F[te_], yw[te_])
        rows[tag].append(s_); sc.append(s_)
    rows["B − A"].append(sc[1] - sc[0])

print(f"{'':20s} {'prosek':>8s} {'sd':>7s} {'min':>8s} {'max':>8s}")
for k, v in rows.items():
    v = np.array(v)
    print(f"{k:20s} {v.mean():8.3f} {v.std():7.3f} {v.min():8.3f} {v.max():8.3f}")

d = np.array(rows["B − A"])
print(f"\nB iznad A na testu: {(d > 0).sum()}/{len(d)} semena")
print(f"prijavljena podela (SEED = {SEED}): A {rows['A standardna'][SEED]:.3f}   "
      f"B {rows['B standardna+DWT'][SEED]:.3f}   B−A {d[SEED]:+.3f}")

In [ ]:
sub_acc = {}
for k, s in enumerate(SUB):
    Fk = np.concatenate([rel[k], ent[k]], axis=1).astype(np.float32)
    a  = np.array([GaussianNB().fit(Fk[tr], yw[tr]).score(Fk[te], yw[te]) for tr, te in LOSO])
    sub_acc[s] = a
    print(f"  {s:3s}  LOSO {a.mean():.3f} ± {a.std():.3f}")

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
m = [sub_acc[s].mean() for s in SUB]
e = [sub_acc[s].std() for s in SUB]
ax.errorbar(range(len(SUB)), m, yerr=e, marker="o", ms=7, lw=2, color=ACCENT, capsize=4)
ax.axhline(CHANCE, lw=1, color=MUTED, ls="--")
ax.set_xticks(range(len(SUB)))
ax.set_xticklabels(SUB)
ax.set_ylabel("tačnost (LOSO)")
ax.set_xlabel("podopseg")
ax.set_title("Tačnost po podopsegu — isprekidano: većinska klasa")

In [ ]:
# Matrica konfuzije za oba skupa i njihova razlika: ne samo koliko, nego GDE se predikcije
# menjaju kad se DWT koeficijenti dodaju. Redovi su tačne oznake, kolone predviđene,
# normalizovano po redu (dakle po klasi), pa je dijagonala osetljivost po zadatku.
labels, cms, acc_ = sorted(set(yw)), {}, {}
for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all)]:
    pr = np.empty_like(yw)
    for tr, te in LOSO:
        pr[te] = GaussianNB().fit(F[tr], yw[tr]).predict(F[te])
    cms[tag] = confusion_matrix(yw, pr, labels=labels, normalize="true")
    acc_[tag] = (pr == yw).mean()

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.4), layout="constrained")
for ax, tag in zip(axes[:2], cms):
    ConfusionMatrixDisplay(cms[tag], display_labels=labels).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(f"{tag} — ukupno {acc_[tag]:.3f}")
    ax.set_xlabel("predviđeno"); ax.set_ylabel("stvarno")
    ax.tick_params(axis="x", rotation=45); ax.grid(False)

Dm = cms["B standardna+DWT"] - cms["A standardna"]
im = axes[2].imshow(Dm, cmap="RdBu_r", vmin=-0.25, vmax=0.25)
for i_ in range(len(labels)):
    for j_ in range(len(labels)):
        if abs(Dm[i_, j_]) >= 0.01:
            axes[2].text(j_, i_, f"{Dm[i_, j_]:+.2f}", ha="center", va="center", fontsize=8,
                         color="white" if abs(Dm[i_, j_]) > 0.15 else "black")
axes[2].set_xticks(range(len(labels))); axes[2].set_xticklabels(labels, rotation=45, ha="right")
axes[2].set_yticks(range(len(labels))); axes[2].set_yticklabels(labels)
axes[2].set_xlabel("predviđeno"); axes[2].set_ylabel("stvarno")
axes[2].set_title(f"razlika B − A (ukupno {acc_['B standardna+DWT'] - acc_['A standardna']:+.3f})")
axes[2].grid(False)
fig.colorbar(im, ax=axes[2], shrink=0.7)

print("promene na dijagonali (osetljivost po zadatku):")
for i_, c_ in enumerate(labels):
    print(f"  {c_:12s} {cms['A standardna'][i_, i_]:.3f} -> "
          f"{cms['B standardna+DWT'][i_, i_]:.3f}   {Dm[i_, i_]:+.3f}")

In [ ]:
a_std = np.array([GaussianNB().fit(F_std[tr], yw[tr]).score(F_std[te], yw[te]) for tr, te in LOSO])
a_all = np.array([GaussianNB().fit(F_all[tr], yw[tr]).score(F_all[te], yw[te]) for tr, te in LOSO])
d = a_all - a_std
from scipy.stats import wilcoxon
print(f"razlika po ispitaniku: {d.mean():+.3f} ± {d.std():.3f}   pozitivnih {(d>0).sum()}/10")
print(wilcoxon(a_all, a_std))

In [ ]:
F_nob = np.concatenate(blocks[len(BANDS):], axis=1).astype(np.float32)   # bez band_*
F_nob_dwt = np.concatenate([F_nob, F_dwt], axis=1)

res = {}
for tag, F in [("bez band_*", F_nob), ("bez band_* + DWT", F_nob_dwt),
               ("A standardna", F_std), ("B standardna+DWT", F_all)]:
    res[tag] = np.array([GaussianNB().fit(F[tr], yw[tr]).score(F[te], yw[te])
                         for tr, te in LOSO])
    print(f"{tag:20s} {F.shape[1]:5d} obeležja   LOSO {res[tag].mean():.3f} ± {res[tag].std():.3f}")

for a, b in [("bez band_*", "bez band_* + DWT"), ("A standardna", "B standardna+DWT")]:
    d = res[b] - res[a]
    print(f"\n{b} − {a}: {d.mean():+.3f} ± {d.std():.3f}  pozitivnih {(d>0).sum()}/10  "
          f"p={wilcoxon(res[b], res[a]).pvalue:.3f}")

In [ ]:
loc = []
for c in coeffs:                                   # c: (n, len_k, 87)
    a = np.abs(c)
    q = np.sort(c ** 2, axis=1)[:, ::-1, :]        # energija, opadajuće
    k = max(1, int(0.1 * c.shape[1]))
    loc += [a.var(axis=1),                                        # rasipanje |koef.|
            a.max(axis=1),                                        # najveći koeficijent
            q[:, :k, :].sum(axis=1) / np.maximum(q.sum(axis=1), 1e-12)]   # koncentracija

F_loc  = np.concatenate(loc, axis=1).astype(np.float32)
F_all2 = np.concatenate([F_std, F_loc], axis=1)
LOC_NAMES = [f"{p}_{s}:{r}" for s in SUB for p in ("V", "M", "K") for r in names]

for tag, F in [("A standardna", F_std), ("B2 standardna+DWT_lok", F_all2)]:
    res[tag] = np.array([GaussianNB().fit(F[tr], yw[tr]).score(F[te], yw[te])
                         for tr, te in LOSO])
    print(f"{tag:24s} {F.shape[1]:5d}   LOSO {res[tag].mean():.3f} ± {res[tag].std():.3f}")
d = res["B2 standardna+DWT_lok"] - res["A standardna"]
print(f"razlika: {d.mean():+.3f} ± {d.std():.3f}  pozitivnih {(d>0).sum()}/10  "
      f"p={wilcoxon(res['B2 standardna+DWT_lok'], res['A standardna']).pvalue:.3f}")

In [ ]:
# Zadatak izričito traži da se izbor pravi na validacionom skupu; C je do sada bio upisan
# ručno. Bira se SAMO nad skupom A pa primenjuje na oba, da poređenje A↔B ne dobije
# prednost od dvostrukog podešavanja.
CS, val_c = (0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0), {}
print(f"{'C':>8s} {'val A':>7s} {'val B':>7s}")
for Cv in CS:
    sc = []
    for F in (F_std, F_all):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=Cv))
        sc.append(m.fit(F[IDX["train"]], yw[IDX["train"]]).score(F[IDX["val"]], yw[IDX["val"]]))
    val_c[Cv] = sc
    print(f"{Cv:8.3f} {sc[0]:7.3f} {sc[1]:7.3f}", flush=True)

C_BEST = max(CS, key=lambda cc: val_c[cc][0])                 # bira se po skupu A
print(f"\nizabrano C = {C_BEST}  (po val skupu A)\n")

lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=C_BEST))
for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all)]:
    a = np.array([lr().fit(F[tr], yw[tr]).score(F[te], yw[te]) for tr, te in LOSO])
    t = lr().fit(F[IDX["train"]], yw[IDX["train"]]).score(F[IDX["test"]], yw[IDX["test"]])
    res["LR " + tag] = a
    print(f"LR {tag:20s} test {t:.3f}   LOSO {a.mean():.3f} ± {a.std():.3f}   "
          f"opseg {a.min():.2f}–{a.max():.2f}")

d = res["LR B standardna+DWT"] - res["LR A standardna"]
print(f"\nLR razlika: {d.mean():+.3f} ± {d.std():.3f}  pozitivnih {(d > 0).sum()}/{len(d)}  "
      f"p={wilcoxon(res['LR B standardna+DWT'], res['LR A standardna']).pvalue:.3f}")

In [ ]:
# Naivni Bayes pretpostavlja nezavisnost obeležja, a skup B ima 1653 međusobno korelisana
# obeležja na 357 prozora. Selekcija po F-vrednosti radi se UNUTAR svakog folda — ako se
# radi jednom nad celim skupom, u ocenu curi informacija iz test folda.
KS = [10, 25, 50, 100, 200, 400, 800, None]
sel_curve = {}
print(f"{'k':>5s} " + "  ".join(f"{t:>17s}" for t in ("A standardna", "B standardna+DWT")) + "   B-A")
for k in KS:
    row = []
    for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all)]:
        a = []
        for tr, te in LOSO:
            if k is None:
                sub = slice(None)
            else:
                Fs_, _ = f_classif(F[tr], yw[tr])
                sub = np.argsort(-np.nan_to_num(Fs_))[:k]
            a.append(GaussianNB().fit(F[tr][:, sub], yw[tr]).score(F[te][:, sub], yw[te]))
        sel_curve[tag, k] = np.array(a)
        row.append(f"{np.mean(a):.3f} ± {np.std(a):.2f}")
    d = sel_curve["B standardna+DWT", k] - sel_curve["A standardna", k]
    print(f"{str(k):>5s} " + "  ".join(f"{v:>17s}" for v in row) + f"   {d.mean():+.3f} "
          f"p={wilcoxon(sel_curve['B standardna+DWT', k], sel_curve['A standardna', k]).pvalue:.3f}",
          flush=True)

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
xs = [k if k else len(ALL_NAMES) for k in KS]
for tag, mk in (("A standardna", "o"), ("B standardna+DWT", "s")):
    ax.errorbar(xs, [sel_curve[tag, k].mean() for k in KS],
                yerr=[sel_curve[tag, k].std() for k in KS], marker=mk, ms=6, lw=1.8, capsize=3,
                color=ACCENT if tag.startswith("A") else MUTED, label=tag)
ax.set_xscale("log"); ax.axhline(CHANCE, lw=1, color=MUTED, ls="--")
ax.set_xlabel("broj zadržanih obeležja (F-selekcija unutar folda)")
ax.set_ylabel("tačnost (LOSO)"); ax.legend(frameon=False, fontsize=9)
ax.set_title("Uticaj dimenzionalnosti na Naivni Bayes")

In [ ]:
# "U kojoj meri" znači i po zadatku, ne samo ukupno: DWT može da pomogne nekim klasama a
# odmogne drugima, što se u zbirnoj tačnosti poništi.
pred = {}
for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all)]:
    p = np.empty_like(yw)
    for tr, te in LOSO:
        p[te] = GaussianNB().fit(F[tr], yw[tr]).predict(F[te])
    pred[tag] = p

print(f"{'zadatak':12s} {'n':>4s} {'A':>7s} {'B':>7s} {'B-A':>8s}")
for c in sorted(set(yw)):
    m  = yw == c
    ra = (pred["A standardna"][m] == c).mean()
    rb = (pred["B standardna+DWT"][m] == c).mean()
    print(f"{c:12s} {m.sum():4d} {ra:7.3f} {rb:7.3f} {rb - ra:+8.3f}")
print(f"{'makro F1':12s} {'':4s} {f1_score(yw, pred['A standardna'], average='macro'):7.3f} "
      f"{f1_score(yw, pred['B standardna+DWT'], average='macro'):7.3f}")
print(f"{'ukupno':12s} {len(yw):4d} {(pred['A standardna'] == yw).mean():7.3f} "
      f"{(pred['B standardna+DWT'] == yw).mean():7.3f}")

In [ ]:
# Predikcija je po prozoru, a svi prozori jednog runa nose istu tačnu oznaku. Agregacija po
# runu nije curenje nego pravilo odlučivanja: prozori se ne mešaju između runova, a ceo run
# ostaje unutar jednog folda jer je podela po ispitanicima. Dva pravila: većinsko glasanje
# nad predikcijama i prosek klasnih verovatnoća.
run_id = srcw[:, 0]                                  # indeks runa za svaki prozor
CLS_NB = np.unique(yw)
RUN_Y  = np.array([yw[run_id == r_][0] for r_ in np.unique(run_id)])
print("prozora po runu:", dict(sorted(collections.Counter(
      collections.Counter(run_id).values()).items())), f"   runova: {len(RUN_Y)}")
print("runova po klasi:", dict(sorted(collections.Counter(RUN_Y).items(), key=lambda t: -t[1])))
print(f"granica po prozoru {CHANCE:.3f}   granica po runu "
      f"{max(collections.Counter(RUN_Y).values()) / len(RUN_Y):.3f}\n")

print(f"{'skup':20s} {'po prozoru':>12s} {'run: glasanje':>15s} {'run: verovatnoće':>18s}")
agg = {}
for tag, F in [("A standardna", F_std), ("B standardna+DWT", F_all)]:
    aw, av, ap = [], [], []
    for tr, te in LOSO:
        m   = GaussianNB().fit(F[tr], yw[tr])
        pw  = m.predict(F[te])
        prb = m.predict_proba(F[te])
        aw.append((pw == yw[te]).mean())
        rid, hv, hp = run_id[te], 0, 0
        for r_ in np.unique(rid):
            sel  = rid == r_
            true = yw[te][sel][0]
            hv  += collections.Counter(pw[sel]).most_common(1)[0][0] == true
            hp  += CLS_NB[prb[sel].mean(0).argmax()] == true
        n_ = len(np.unique(rid))
        av.append(hv / n_); ap.append(hp / n_)
    agg[tag] = {"prozor": np.array(aw), "glas": np.array(av), "verov": np.array(ap)}
    print(f"{tag:20s} {np.mean(aw):.3f} ± {np.std(aw):.2f} "
          f"{np.mean(av):9.3f} ± {np.std(av):.2f} {np.mean(ap):12.3f} ± {np.std(ap):.2f}")

print()
for tag in agg:
    d = agg[tag]["verov"] - agg[tag]["prozor"]
    print(f"{tag:20s} verovatnoće − prozor: {d.mean():+.3f} ± {d.std():.3f}  "
          f"pozitivnih {(d > 0).sum()}/{len(d)}  "
          f"p={wilcoxon(agg[tag]['verov'], agg[tag]['prozor']).pvalue:.3f}")
d = agg["B standardna+DWT"]["verov"] - agg["A standardna"]["verov"]
print(f"\nB − A po runu (verovatnoće): {d.mean():+.3f} ± {d.std():.3f}  "
      f"p={wilcoxon(agg['B standardna+DWT']['verov'], agg['A standardna']['verov']).pvalue:.3f}")

## HMMN rekreacija

In [ ]:
NREG, TW = Xw.shape[2], Xw.shape[1]      # R je u ćeliji 16 prevezan na listu desnih regiona
NYQ = 1 / (2 * TR)

base = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=0.1))
EYE  = np.eye(NREG, dtype=bool)
# Za razliku od ćelije 23, koja obeležja gradi od DWT koeficijenata, HMMN radi nad
# REKONSTRUISANIM signalom podopsega: mesh se procenjuje nad nizovima pune dužine TW,
# pa su svi podopsezi uporedivi. A_l = A_{l+1} + D_{l+1}, pa jedna dekompozicija daje svih 2L+1.
c, XS = pywt.wavedec(Xw, WAVELET, level=L, axis=1), {}      # [cA_L, cD_L, ..., cD_1]
for k in range(L + 1):
    z = [ci if j == k else np.zeros_like(ci) for j, ci in enumerate(c)]
    XS[f"A{L}" if k == 0 else f"D{L - k + 1}"] = \
        pywt.waverec(z, WAVELET, axis=1)[:, :TW, :].astype(np.float32)
for l in range(L - 1, 0, -1):
    XS[f"A{l}"] = XS[f"A{l + 1}"] + XS[f"D{l + 1}"]
XS["A0"] = XS["A1"] + XS["D1"]                              # = originalni signal

HBAND = ["A0"] + [f"A{l}" for l in range(1, L + 1)] + [f"D{l}" for l in range(1, L + 1)]
print(f"{len(HBAND)} podopsega (2L+1, L={L}):", HBAND)
print(f"rekonstrukcija A0 − original, maks. odstupanje: {np.abs(XS['A0'] - Xw).max():.2e}")
for s in HBAND[1:]:
    k = int(s[1:])
    lo, hi = (0, NYQ / 2 ** k) if s[0] == "A" else (NYQ / 2 ** k, NYQ / 2 ** (k - 1))
    print(f"  {s:3s}  {lo:.3f}–{hi:.3f} Hz")

In [ ]:
# Produžena mreža + granica λ→∞ (a ∝ G[s,r], korelacija na p suseda).
# Ako mesh ne nadmaši tu granicu, rebrasta regresija ne donosi ništa pri TW=176.
Z0 = XS["A0"] - XS["A0"].mean(1, keepdims=True)
Z0 = Z0 / np.where(Z0.std(1, keepdims=True) > 0, Z0.std(1, keepdims=True), 1)
G0 = np.einsum("ntr,nts->nrs", Z0, Z0)                      # (n, R, R) Gramove matrice

for pm in (10, 20, 40):
    NB = np.argpartition(-np.where(EYE, -np.inf, G0), pm, axis=2)[:, :, :pm]
    for lam in (512.0, 2048.0, 8192.0, 32768.0):
        M, Ip = np.zeros((len(Xw), NREG, NREG), np.float32), lam * np.eye(pm)
        for i in range(len(Xw)):
            for r in range(NREG):
                s = NB[i, r]
                M[i, r, s] = np.linalg.solve(G0[i][np.ix_(s, s)] + Ip, G0[i][s, r])
        M = M.reshape(len(M), -1)
        print(f"  p={pm:2d}  λ={lam:6.0f}   val "
              f"{clone(base).fit(M[IDX['train']], yw[IDX['train']]).score(M[IDX['val']], yw[IDX['val']]):.3f}",
              flush=True)
    M = np.zeros((len(Xw), NREG, NREG), np.float32)
    for i in range(len(Xw)):
        for r in range(NREG):
            s = NB[i, r]
            M[i, r, s] = G0[i][s, r]
    M = M.reshape(len(M), -1)
    print(f"  p={pm:2d}  λ→∞         val "
          f"{clone(base).fit(M[IDX['train']], yw[IDX['train']]).score(M[IDX['val']], yw[IDX['val']]):.3f}\n",
          flush=True)

In [ ]:
P_MESH, LAM = 10, 2048.0                  # upisati najbolji par iz ćelije iznad
Ip, iu = LAM * np.eye(P_MESH), np.triu_indices(NREG, 1)
FEAT = {"mesh": {}, "korel": {}, "nizovi": {}}

for b in HBAND:
    Z = XS[b] - XS[b].mean(1, keepdims=True)
    Z = Z / np.where(Z.std(1, keepdims=True) > 0, Z.std(1, keepdims=True), 1)
    G = np.einsum("ntr,nts->nrs", Z, Z)
    NB = np.argpartition(-np.where(EYE, -np.inf, G), P_MESH, axis=2)[:, :, :P_MESH]
    M = np.zeros((len(Xw), NREG, NREG), np.float32)
    for i in range(len(Xw)):
        for r in range(NREG):
            s = NB[i, r]
            M[i, r, s] = np.linalg.solve(G[i][np.ix_(s, s)] + Ip, G[i][s, r])
    FEAT["mesh"][b]   = M.reshape(len(M), -1)               # graf je usmeren -> cela R×R
    FEAT["korel"][b]  = (G / TW)[:, iu[0], iu[1]].astype(np.float32)
    FEAT["nizovi"][b] = XS[b].reshape(len(XS[b]), -1)
    print(f"  {b:3s} gotovo", flush=True)

print({k: FEAT[k]["A0"].shape[1] for k in FEAT}, "obeležja po podopsegu")

In [ ]:
FEAT["mesh-OMP"] = {}
for b in HBAND:
    Z = XS[b] - XS[b].mean(1, keepdims=True)
    Z = Z / np.where(Z.std(1, keepdims=True) > 0, Z.std(1, keepdims=True), 1)
    Cb = (np.einsum("ntr,nts->nrs", Z, Z) / TW).astype(np.float64)
    M  = np.zeros((len(Xw), NREG, NREG), np.float32)
    for i in range(len(Xw)):
        C = Cb[i]
        for r in range(NREG):
            S, rsd = [], C[:, r].copy(); rsd[r] = 0.0
            for _ in range(P_MESH):
                S.append(int(np.argmax(np.abs(rsd))))
                aa  = np.linalg.solve(C[np.ix_(S, S)] + (LAM / TW) * np.eye(len(S)), C[S, r])
                rsd = C[:, r] - C[:, S] @ aa
                rsd[r] = 0.0; rsd[S] = 0.0
            M[i, r, S] = aa
    FEAT["mesh-OMP"][b] = M.reshape(len(M), -1)
    print(f"  {b:3s} gotovo", flush=True)

In [ ]:
# Jedan prolaz daje i tačnost baznog klasifikatora i verovatnoće koje meta sloj traži.
# Meta se uči na out-of-fold verovatnoćama, podelom po ispitanicima UNUTAR trening skupa —
# inače bi učio na precenjenim verovatnoćama i fuzija bi bila lažno dobra.
CLS, PTR, PTE, WT, base_acc = np.unique(yw), {}, {}, {}, {}
NC = len(CLS)

for k in FEAT:
    for b in HBAND:
        F, a = FEAT[k][b], []
        for fi, (tr, te) in enumerate(LOSO):
            P = np.zeros((len(tr), NC))
            for itr, ite in GroupKFold(n_splits=3).split(tr, yw[tr], gw[tr]):
                P[ite] = clone(base).fit(F[tr][itr], yw[tr][itr]).predict_proba(F[tr][ite])
            m = clone(base).fit(F[tr], yw[tr])
            PTR[k, b, fi] = P
            PTE[k, b, fi] = m.predict_proba(F[te])
            WT[k, b, fi]  = m.score(F[tr], yw[tr])          # težina za WMV
            a.append(m.score(F[te], yw[te]))
        base_acc[k, b] = np.array(a)
        print(f"  {k:6s} {b:3s}  LOSO {base_acc[k, b].mean():.3f} ± {base_acc[k, b].std():.3f}",
              flush=True)

print(f"\n{'nivo':5s} " + "  ".join(f"{k:>13s}" for k in FEAT))
for b in HBAND:
    print(f"{b:5s} " + "  ".join(
        f"{base_acc[k, b].mean():.3f}±{base_acc[k, b].std():.2f}".rjust(13) for k in FEAT))
print(f"{'većina':5s} {CHANCE:13.3f}")

In [ ]:
# Bazne verovatnoće su već izračunate, pa je fuzija za bilo koji podskup samo nadovezivanje.
# Rad varira E od 2 do 2L; podopsezi se dodaju opadajuće po sopstvenoj tačnosti.
fus = {}
for k in FEAT:
    order = sorted(HBAND, key=lambda b: -base_acc[k, b].mean())
    print(f"\n{k}   redosled: {order}")
    print(f"{'E':>3s} " + "  ".join(f"{c:>13s}" for c in ("FSG-L", "FSG-S", "MV", "WMV")))
    for E in range(2, len(HBAND) + 1):
        acc = {c: [] for c in ("FSG-L", "FSG-S", "MV", "WMV")}
        for fi, (tr, te) in enumerate(LOSO):
            sel = order[:E]
            Ptr = np.hstack([PTR[k, b, fi] for b in sel])   # (n_tr, C·E)
            Pte = np.hstack([PTE[k, b, fi] for b in sel])
            Q   = np.stack([PTE[k, b, fi] for b in sel])    # (E, n_te, C)
            w   = np.array([WT[k, b, fi] for b in sel])
            acc["FSG-L"].append(clone(base).fit(Ptr, yw[tr]).score(Pte, yw[te]))
            acc["FSG-S"].append(make_pipeline(StandardScaler(), LinearSVC(C=1.0))
                                .fit(Ptr, yw[tr]).score(Pte, yw[te]))
            v = np.zeros((len(te), NC))
            for e in range(E):
                v[np.arange(len(te)), Q[e].argmax(1)] += 1
            acc["MV"].append((CLS[v.argmax(1)] == yw[te]).mean())
            acc["WMV"].append((CLS[(Q * w[:, None, None]).sum(0).argmax(1)] == yw[te]).mean())
        fus[k, E] = {c: np.array(v) for c, v in acc.items()}
        print(f"{E:3d} " + "  ".join(f"{fus[k, E][c].mean():.3f}±{fus[k, E][c].std():.2f}".rjust(13)
                                     for c in ("FSG-L", "FSG-S", "MV", "WMV")), flush=True)

In [ ]:
# Rad: tačnost A_l opada sa nivoom (do šanse na l=11), tačnost D_l raste do l=5–6.
# Kod nas L=4, pa se vidi samo početak tog trenda.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True, layout="constrained")
for ax, k in zip(axes, FEAT):
    for pref, mk, col in (("A", "o", ACCENT), ("D", "s", MUTED)):
        lv = list(range(0 if pref == "A" else 1, L + 1))
        ax.errorbar(lv, [base_acc[k, f"{pref}{l}"].mean() for l in lv],
                    yerr=[base_acc[k, f"{pref}{l}"].std() for l in lv],
                    marker=mk, ms=6, lw=1.8, capsize=3, color=col, label=f"{pref}$_l$")
    ax.axhline(fus[k, len(HBAND)]["FSG-S"].mean(), lw=1.2, color=ACCENT, ls=":", label="FSG, svi")
    ax.axhline(CHANCE, lw=1, color=MUTED, ls="--", label="većinska klasa")
    ax.set_title(k); ax.set_xlabel("nivo dekompozicije $l$"); ax.set_xticks(range(L + 1))
axes[0].set_ylabel("tačnost (LOSO)"); axes[0].legend(fontsize=8, frameon=False)
fig.suptitle("HMMN — tačnost po podopsegu i posle fuzije", x=0.01, ha="left")

In [ ]:
# Konfiguracija iz rada, izabrana pre gledanja rezultata: mesh arc weights + FSG preko svih
# podopsega, meta sloj linearni SVM. NIJE najbolja od 96 kombinacija iz prethodne ćelije —
# mesh sam ima vrh 0.866 (FSG-L, E=6) — i namerno se ne prijavljuje ta, da izbor ne bi bio
# post hoc.
hmmn_rad = fus["mesh", len(HBAND)]["FSG-S"]
print(f"HMMN (rad: mesh + FSG-S, E={len(HBAND)})   LOSO {hmmn_rad.mean():.3f} ± {hmmn_rad.std():.3f}\n")
for tag in ("A standardna", "B standardna+DWT", "LR A standardna", "LR B standardna+DWT"):
    d = hmmn_rad - res[tag]
    print(f"HMMN − {tag:22s} {d.mean():+.3f} ± {d.std():.3f}  "
          f"pozitivnih {(d > 0).sum()}/{len(d)}  p={wilcoxon(hmmn_rad, res[tag]).pvalue:.3f}")

In [ ]:
# Popravka koja sledi iz dijagnoze: ako mesh pada zbog uslovljenosti, menja se IZBOR SUSEDA,
# a ne regresija. Tri pravila, isto p i isto λ, razlikuje se samo susedstvo:
#   top-p  — p najkorelisanijih (pravilo iz rada; MORA da reprodukuje 0.779, to je provera)
#   OMP    — pohlepno po doprinosu ostatku; svaki sused nosi novu informaciju
#   τ=…    — pohlepno po korelaciji sa r, uz zabranu suseda korelisanijih od τ sa već izabranima
Za = XS["A0"] - XS["A0"].mean(1, keepdims=True)
Za = Za / np.where(Za.std(1, keepdims=True) > 0, Za.std(1, keepdims=True), 1)
Ca  = (np.einsum("ntr,nts->nrs", Za, Za) / TW).astype(np.float64)
RID = (LAM / TW) * np.eye(P_MESH)

sel_acc = {}
for rule in ("top-p", "OMP", "τ=0.4", "τ=0.5", "τ=0.6"):
    Ms, cond_r = np.zeros((len(Xw), NREG, NREG), np.float32), []
    for i in range(len(Xw)):
        C = Ca[i]
        for r in range(NREG):
            if rule == "top-p":
                cr = C[:, r].copy(); cr[r] = -np.inf
                S = list(np.argpartition(-cr, P_MESH)[:P_MESH])
            elif rule == "OMP":
                S, rsd = [], C[:, r].copy(); rsd[r] = 0.0
                for _ in range(P_MESH):
                    S.append(int(np.argmax(np.abs(rsd))))
                    aa  = np.linalg.solve(C[np.ix_(S, S)] + (LAM / TW) * np.eye(len(S)), C[S, r])
                    rsd = C[:, r] - C[:, S] @ aa
                    rsd[r] = 0.0; rsd[S] = 0.0
            else:
                tau, S = float(rule.split("=")[1]), []
                order = np.argsort(-C[:, r])
                blocked = np.zeros(NREG, bool); blocked[r] = True
                for j in order:
                    if blocked[j]:
                        continue
                    S.append(int(j))
                    if len(S) == P_MESH:
                        break
                    blocked |= np.abs(C[:, j]) > tau          # kolinearnost je bez znaka
                for j in order:                               # dopuna ako je τ presekao previše
                    if len(S) == P_MESH:
                        break
                    if j != r and int(j) not in S:
                        S.append(int(j))
            Ms[i, r, S] = np.linalg.solve(C[np.ix_(S, S)] + RID, C[S, r])
            cond_r.append(np.linalg.cond(C[np.ix_(S, S)]))

    Fs = Ms.reshape(len(Ms), -1)
    a  = np.array([clone(base).fit(Fs[tr], yw[tr]).score(Fs[te], yw[te]) for tr, te in LOSO])
    sel_acc[rule] = a
    d = a - base_acc["korel", "A0"]
    print(f"  {rule:6s}  cond {np.median(cond_r):7.1f}   LOSO {a.mean():.3f} ± {a.std():.3f}   "
          f"mesh−korel {d.mean():+.3f}  p={wilcoxon(a, base_acc['korel','A0']).pvalue:.3f}",
          flush=True)

# --- posle petlje, neuvučeno ----------------------------------------------------------
print()
for rl in sel_acc:                              # drugo ime, da ne gazi `rule` iz petlje
    d   = sel_acc[rl] - base_acc["korel", "A0"]
    mde = (2.262 + 0.883) * d.std(ddof=1) / np.sqrt(len(d))   # dvostrani 0.05, snaga 0.80, n=10
    print(f"{rl:6s}  d = {d.mean():+.3f} ± {d.std(ddof=1):.3f}   "
          f"najmanji detektabilan efekat ≈ {mde:.3f}")

if {"OMP", "top-p"} <= sel_acc.keys():
    d = sel_acc["OMP"] - sel_acc["top-p"]
    print(f"\nOMP − top-p: {d.mean():+.3f} ± {d.std():.3f}  "
          f"p={wilcoxon(sel_acc['OMP'], sel_acc['top-p']).pvalue:.3f}")
else:
    print("\nnedostaje:", {"OMP", "top-p"} - sel_acc.keys(), " izvršeno:", list(sel_acc))

In [ ]:
# Hipoteza: lučna težina je ocena veće varijanse od korelacije, pa joj treba duži prozor
# (rad ima 1940 tačaka, ovde 176). Mora se meriti pri λ gde regresija još radi: pri λ=2048
# je mesh već degenerisan u korelaciju na p suseda (vidi ćeliju sa mrežom λ), pa bi se
# poredila korelacija sa korelacijom. λ=32 je vrednost iz rada.
# Poredi se razlika UNUTAR svake dužine — sa TW se menjaju i broj prozora i odnos klasa.
for W in (88, 132, 176):
    Xv, yv, gv = [], [], []
    for i, rn in enumerate(runs):
        st = list(range(0, rn.shape[0] - W + 1, W // 2))
        if st[-1] != rn.shape[0] - W:
            st.append(rn.shape[0] - W)
        for s in st:
            seg = rn[s:s + W]
            Xv.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
            yv.append(y[i]); gv.append(subjects[i])
    Xv, yv, gv = np.stack(Xv).astype(np.float32), np.array(yv), np.array(gv)

    Zv = Xv - Xv.mean(1, keepdims=True)
    Zv = Zv / np.where(Zv.std(1, keepdims=True) > 0, Zv.std(1, keepdims=True), 1)
    Gv  = np.einsum("ntr,nts->nrs", Zv, Zv)
    NBv = np.argpartition(-np.where(EYE, -np.inf, Gv), P_MESH, axis=2)[:, :, :P_MESH]
    lo  = list(LeaveOneGroupOut().split(Xv, yv, gv))

    Fk = (Gv / W)[:, iu[0], iu[1]].astype(np.float32)
    ak = np.array([clone(base).fit(Fk[tr], yv[tr]).score(Fk[te], yv[te]) for tr, te in lo])
    print(f"TW={W:3d}  n={len(Xv):3d}   korel {ak.mean():.3f} ± {ak.std():.3f}")

    for lam in (8.0, 32.0, 128.0, 2048.0):
        Ipv = lam * np.eye(P_MESH)
        Mv  = np.zeros((len(Xv), NREG, NREG), np.float32)
        for i in range(len(Xv)):
            for r in range(NREG):
                s = NBv[i, r]
                Mv[i, r, s] = np.linalg.solve(Gv[i][np.ix_(s, s)] + Ipv, Gv[i][s, r])
        Fm = Mv.reshape(len(Mv), -1)
        am = np.array([clone(base).fit(Fm[tr], yv[tr]).score(Fm[te], yv[te]) for tr, te in lo])
        d  = am - ak
        print(f"   λ={lam:6.0f}  mesh {am.mean():.3f} ± {am.std():.3f}   "
              f"mesh−korel {d.mean():+.3f} ± {d.std():.3f}  p={wilcoxon(am, ak).pvalue:.3f}",
              flush=True)
    print(flush=True)

In [ ]:
# Zašto duži prozor ne pomaže: susedi su p najkorelisanijih regiona, dakle i međusobno
# kolinearni, pa je C_ss loše uslovljena. Kolinearnost je svojstvo populacije, ne uzorka —
# ako je stvarna, uslovljenost ostaje ravna sa TW; ako je artefakt izbora suseda iz šuma,
# opada. To razlikuje dva objašnjenja.
for W in (88, 132, 176):
    Xv = []
    for rn in runs:
        st = list(range(0, rn.shape[0] - W + 1, W // 2))
        if st[-1] != rn.shape[0] - W:
            st.append(rn.shape[0] - W)
        for s in st:
            seg = rn[s:s + W]
            Xv.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
    Xv = np.stack(Xv).astype(np.float32)

    Zv = Xv - Xv.mean(1, keepdims=True)
    Zv = Zv / np.where(Zv.std(1, keepdims=True) > 0, Zv.std(1, keepdims=True), 1)
    Gv  = np.einsum("ntr,nts->nrs", Zv, Zv) / W          # korelaciona matrica
    NBv = np.argpartition(-np.where(EYE, -np.inf, Gv), P_MESH, axis=2)[:, :, :P_MESH]

    ut, cond, coll = np.triu_indices(P_MESH, 1), [], []
    for i in range(0, len(Xv), 3):                        # svaki treći prozor, dovoljno
        for r in range(NREG):
            Css = Gv[i][np.ix_(NBv[i, r], NBv[i, r])]
            cond.append(np.linalg.cond(Css))
            coll.append(np.abs(Css[ut]).mean())
    print(f"TW={W:3d}   cond(C_ss): medijana {np.median(cond):7.1f}  90% {np.percentile(cond, 90):8.1f}"
          f"   |korelacija| među susedima: {np.mean(coll):.3f}", flush=True)

In [ ]:
# Kontrola: fazna randomizacija čuva spektar (dakle i autokorelaciju) svakog regiona, a
# nezavisnom fazom po regionu razbija spregu MEĐU regionima. Ako je cond ≈ 31 posledica
# stvarne kolinearnosti, ovde mora da padne i da postane zavisna od TW; ako je artefakt
# argmax izbora suseda iz šuma, ostaće ista.
rng_pr, runs_pr = np.random.default_rng(0), []
for rn in runs:
    Tr = rn.shape[0]
    Fr = np.fft.rfft(rn - rn.mean(0), axis=0)          # (Tr//2+1, NREG)
    ph = rng_pr.uniform(0, 2 * np.pi, Fr.shape)        # nezavisna faza po regionu
    ph[0] = 0.0                                        # DC mora ostati realan
    if Tr % 2 == 0:
        ph[-1] = 0.0                                   # kao i Nyquist, za parno Tr
    runs_pr.append(np.fft.irfft(np.abs(Fr) * np.exp(1j * ph), n=Tr, axis=0).astype(np.float32))

ut = np.triu_indices(P_MESH, 1)
for tag, src in (("original", runs), ("fazno rand.", runs_pr)):
    for W in (88, 132, 176):
        Xv = []
        for rn in src:
            st = list(range(0, rn.shape[0] - W + 1, W // 2))
            if st[-1] != rn.shape[0] - W:
                st.append(rn.shape[0] - W)
            for s in st:
                seg = rn[s:s + W]
                Xv.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
        Xv = np.stack(Xv).astype(np.float32)

        Zv = Xv - Xv.mean(1, keepdims=True)
        Zv = Zv / np.where(Zv.std(1, keepdims=True) > 0, Zv.std(1, keepdims=True), 1)
        Gv  = np.einsum("ntr,nts->nrs", Zv, Zv) / W
        NBv = np.argpartition(-np.where(EYE, -np.inf, Gv), P_MESH, axis=2)[:, :, :P_MESH]

        cond, coll = [], []
        for i in range(0, len(Xv), 3):
            for r in range(NREG):
                Css = Gv[i][np.ix_(NBv[i, r], NBv[i, r])]
                cond.append(np.linalg.cond(Css))
                coll.append(np.abs(Css[ut]).mean())
        print(f"{tag:12s} TW={W:3d}   cond: medijana {np.median(cond):7.1f}  "
              f"90% {np.percentile(cond, 90):8.1f}   |korelacija| među susedima: {np.mean(coll):.3f}",
              flush=True)
    print(flush=True)

In [ ]:
# Koliko tačnosti dolazi od poklapanja sa dizajnom, a koliko od reprezentacije: prozori se
# sada uzimaju sa NASUMIČNE pozicije u runu, uz isti broj prozora po runu — dakle isto n i
# isti odnos klasa, pa su brojevi uporedivi sa tabelom po podopsezima. Napomena za tekst:
# EMOTION run ima tačno WIN frejmova, pa se tih 20 prozora ne mogu pomeriti i ostaju
# poravnati; izmereni padovi su zato donja granica.
NREP = 3
jit = {k: [] for k in ("nizovi", "korel", "mesh")}

for rep in range(NREP):
    rg = np.random.default_rng(100 + rep)
    Xj, yj, gj = [], [], []
    for i, rn in enumerate(runs):
        Tr = rn.shape[0]
        st = list(range(0, Tr - WIN + 1, STRIDE))
        if st[-1] != Tr - WIN:
            st.append(Tr - WIN)
        for _ in st:                                   # isti broj prozora, nasumične pozicije
            s = int(rg.integers(0, Tr - WIN + 1))
            seg = rn[s:s + WIN]
            Xj.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
            yj.append(y[i]); gj.append(subjects[i])
    Xj, yj, gj = np.stack(Xj).astype(np.float32), np.array(yj), np.array(gj)

    Zj = Xj - Xj.mean(1, keepdims=True)
    Zj = Zj / np.where(Zj.std(1, keepdims=True) > 0, Zj.std(1, keepdims=True), 1)
    Gj  = np.einsum("ntr,nts->nrs", Zj, Zj)
    NBj = np.argpartition(-np.where(EYE, -np.inf, Gj), P_MESH, axis=2)[:, :, :P_MESH]
    Mj, Ipj = np.zeros((len(Xj), NREG, NREG), np.float32), LAM * np.eye(P_MESH)
    for i in range(len(Xj)):
        for r in range(NREG):
            s = NBj[i, r]
            Mj[i, r, s] = np.linalg.solve(Gj[i][np.ix_(s, s)] + Ipj, Gj[i][s, r])

    Fj = {"nizovi": Xj.reshape(len(Xj), -1),
          "korel":  (Gj / WIN)[:, iu[0], iu[1]].astype(np.float32),
          "mesh":   Mj.reshape(len(Mj), -1)}
    lo = list(LeaveOneGroupOut().split(Xj, yj, gj))
    for k, Fk in Fj.items():
        a = np.array([clone(base).fit(Fk[tr], yj[tr]).score(Fk[te], yj[te]) for tr, te in lo])
        jit[k].append(a.mean())
    print(f"  ponavljanje {rep + 1}/{NREP} gotovo", flush=True)

print(f"\n{'skup':8s} {'poravnato':>10s} {'pomereno':>18s} {'pad':>8s}")
for k in ("nizovi", "korel", "mesh"):
    o, j = base_acc[k, "A0"].mean(), np.array(jit[k])
    print(f"{k:8s} {o:10.3f}   {j.mean():.3f} ± {j.std():.3f}   {j.mean() - o:+8.3f}")
print(f"{'većina':8s} {CHANCE:10.3f}")

In [ ]:
# Provera glavne tvrdnje na pomerenim prozorima. Sirovi nizovi su izgubili 52 poena, a
# standardna obeležja su spektralna i autokorelaciona, dakle po konstrukciji neosetljiva
# na položaj prozora. Ako A i B prežive pomak, glavni nalaz rada stoji na čistom osnovu.
NREP2 = 3
jit2 = {"A standardna": [], "B standardna+DWT": []}
for rep in range(NREP2):
    rg = np.random.default_rng(200 + rep)
    Xj, yj, gj = [], [], []
    for i, rn in enumerate(runs):
        Tr = rn.shape[0]
        st = list(range(0, Tr - WIN + 1, STRIDE))
        if st[-1] != Tr - WIN:
            st.append(Tr - WIN)
        for _ in st:
            s = int(rg.integers(0, Tr - WIN + 1))
            seg = rn[s:s + WIN]
            Xj.append((seg - seg.mean(0)) / np.where(seg.std(0) > 0, seg.std(0), 1))
            yj.append(y[i]); gj.append(subjects[i])
    Xj, yj, gj = np.stack(Xj).astype(np.float32), np.array(yj), np.array(gj)

    ff, PP = welch(Xj, fs=1 / TR, nperseg=WIN, axis=1)
    pp = PP / PP.sum(axis=1, keepdims=True)
    bk = [pp[:, (ff >= lo) & (ff < hi), :].sum(axis=1) for lo, hi in BANDS]
    bk += [-(pp * np.log(pp + 1e-12)).sum(axis=1), (ff[None, :, None] * pp).sum(axis=1),
           (Xj[:, :-1] * Xj[:, 1:]).mean(axis=1), (Xj[:, :-5] * Xj[:, 5:]).mean(axis=1)]
    Fa_ = np.concatenate(bk, axis=1).astype(np.float32)

    cf = pywt.wavedec(Xj, WAVELET, level=L, axis=1)
    en = [(cc ** 2).sum(axis=1) for cc in cf]
    tt = np.maximum(np.sum(en, axis=0), 1e-12)
    rl = [e / tt for e in en]
    et = []
    for cc in cf:
        q = cc ** 2
        q = q / np.maximum(q.sum(axis=1, keepdims=True), 1e-12)
        et.append(-(q * np.log(q + 1e-12)).sum(axis=1))
    Fb_ = np.concatenate([Fa_, np.concatenate(rl + et, axis=1).astype(np.float32)], axis=1)

    lo_ = list(LeaveOneGroupOut().split(Xj, yj, gj))
    for tag, F in (("A standardna", Fa_), ("B standardna+DWT", Fb_)):
        a = np.array([GaussianNB().fit(F[tr], yj[tr]).score(F[te], yj[te]) for tr, te in lo_])
        jit2[tag].append(a.mean())
    print(f"  ponavljanje {rep + 1}/{NREP2} gotovo", flush=True)

print(f"\n{'skup':20s} {'poravnato':>10s} {'pomereno':>18s} {'pad':>8s}")
for tag in ("A standardna", "B standardna+DWT"):
    ref, j = res[tag].mean(), np.array(jit2[tag])
    print(f"{tag:20s} {ref:10.3f}   {j.mean():.3f} ± {j.std():.3f}   {j.mean() - ref:+8.3f}")
print(f"{'većinska klasa':20s} {CHANCE:10.3f}")

In [ ]:
# Naslovna figura rada: šta preživljava nasumičan pomak prozora. Sirovi vremenski nizovi žive
# od poravnanja sa dizajnom paradigme, dok su sva obeležja koja se u radu koriste nezavisna od
# položaja prozora. Stubac „poravnato" je LOSO prosek, stubac „pomereno" je prosek preko tri
# ponavljanja, a njegova greška je rasipanje MEĐU ponavljanjima; rasipanje po ispitanicima
# navedeno je u tabeli ispod figure, jer su to dve različite veličine i ne mešaju se na istoj osi.
SETS = [("sirovi\nnizovi",       base_acc["nizovi", "A0"], np.array(jit["nizovi"])),
        ("parne\nkorelacije",    base_acc["korel",  "A0"], np.array(jit["korel"])),
        ("mesh arc\nweights",    base_acc["mesh",   "A0"], np.array(jit["mesh"])),
        ("A\nstandardna",        res["A standardna"],        np.array(jit2["A standardna"])),
        ("B standardna\n+ DWT",  res["B standardna+DWT"],    np.array(jit2["B standardna+DWT"]))]

x, w = np.arange(len(SETS)), 0.38
fig, ax = plt.subplots(figsize=(9.5, 4.8), layout="constrained")
ax.bar(x - w / 2, [s[1].mean() for s in SETS], w, color=ACCENT, label="poravnati prozori")
ax.bar(x + w / 2, [s[2].mean() for s in SETS], w, yerr=[s[2].std() for s in SETS],
       color=MUTED, capsize=4, label="nasumičan pomak")
for xi, (_, al, ji) in zip(x, SETS):
    d = ji.mean() - al.mean()
    ax.annotate(f"{d:+.3f}", (xi, max(al.mean(), ji.mean()) + 0.035), ha="center",
                fontsize=10, fontweight="bold" if d < -0.05 else "normal",
                color="#b6553b" if d < -0.05 else MUTED)
ax.axhline(CHANCE, lw=1, color=MUTED, ls="--", label="većinska klasa")
ax.set_xticks(x); ax.set_xticklabels([s[0] for s in SETS], fontsize=9)
ax.set_ylim(0, 1.05); ax.set_ylabel("tačnost (LOSO)")
ax.legend(frameon=False, fontsize=9, ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.09))
ax.set_title("Poravnanje sa dizajnom paradigme: šta preživljava nasumičan pomak prozora")

print(f"{'skup obeležja':22s} {'poravnato':>16s} {'pomereno':>16s} {'pad':>8s}")
for nm, al, ji in SETS:
    nm = nm.replace("\n", " ")
    print(f"{nm:22s} {al.mean():.3f} ± {al.std():.3f}   {ji.mean():.3f} ± {ji.std():.3f}   "
          f"{ji.mean() - al.mean():+8.3f}")
print(f"{'većinska klasa':22s} {CHANCE:9.3f}")
print("\n± poravnato = po ispitanicima (LOSO);  ± pomereno = po ponavljanjima")

In [ ]:
# Sveska sprovodi desetine uparenih testova nad istim podacima, pa se p-vrednosti ne smeju
# čitati pojedinačno. Ovde se sva glavna poređenja skupljaju na jedno mesto i uz njih se daje
# Benjamini-Hochberg korekcija (kontrola očekivanog udela lažnih otkrića).
TESTS = [("B − A (NB)",                 res["B standardna+DWT"],   res["A standardna"]),
         ("B − A (LR)",                 res["LR B standardna+DWT"], res["LR A standardna"]),
         ("bez opsega +DWT − bez opsega", res["bez band_* + DWT"],  res["bez band_*"]),
         ("vremenska lokalizacija − A",  res["B2 standardna+DWT_lok"], res["A standardna"]),
         ("B − A pri k=50",             sel_curve["B standardna+DWT", 50], sel_curve["A standardna", 50]),
         ("B − A pri k=400",            sel_curve["B standardna+DWT", 400], sel_curve["A standardna", 400]),
         ("mesh − korel (top-p)",       sel_acc["top-p"],  base_acc["korel", "A0"]),
         ("mesh − korel (OMP)",         sel_acc["OMP"],    base_acc["korel", "A0"]),
         ("mesh − korel (τ=0.4)",       sel_acc["τ=0.4"],  base_acc["korel", "A0"]),
         ("mesh − korel (τ=0.5)",       sel_acc["τ=0.5"],  base_acc["korel", "A0"]),
         ("mesh − korel (τ=0.6)",       sel_acc["τ=0.6"],  base_acc["korel", "A0"]),
         ("HMMN − A (NB)",              hmmn_rad,          res["A standardna"]),
         ("HMMN − A (LR)",              hmmn_rad,          res["LR A standardna"]),
         ("globalni signal − bez obrade", prep["3 + globalni signal"], prep["0 kako jeste"])]

names, dd, pp_ = [], [], []
for nm, a1, a2 in TESTS:
    d_ = np.asarray(a1) - np.asarray(a2)
    names.append(nm); dd.append(d_.mean())
    pp_.append(1.0 if np.allclose(d_, 0) else
               wilcoxon(np.asarray(a1), np.asarray(a2), zero_method="zsplit").pvalue)

pp_ = np.array(pp_); m_ = len(pp_)
o_ = np.argsort(pp_)
q_ = np.minimum.accumulate((pp_[o_] * m_ / np.arange(1, m_ + 1))[::-1])[::-1]
qq = np.empty(m_); qq[o_] = np.minimum(q_, 1.0)

print(f"{'poređenje':32s} {'razlika':>9s} {'p':>8s} {'p (BH)':>9s}")
for i_ in np.argsort(pp_):
    zn = "  *" if qq[i_] < 0.05 else ""
    print(f"{names[i_]:32s} {dd[i_]:+9.3f} {pp_[i_]:8.3f} {qq[i_]:9.3f}{zn}")
print(f"\nukupno poređenja: {m_}   preživljava BH na 0.05: {(qq < 0.05).sum()}")